In [1]:
# from langchain_openai import ChatOpenAI
# from dotenv import load_dotenv
# import os
# load_dotenv()

# llm = ChatOpenAI( model='openai/gpt-4.1',
#     api_key= os.getenv("GITHUB_TOKEN2"),
#     base_url="https://models.github.ai/inference")

# responce = llm.invoke("hi").content 
# print(responce)
# # print(responce)

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
responce = llm.invoke("hi").content 
print(responce)

Hi! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [3]:
import sys
sys.executable


'e:\\CLg Acadamics\\Mega Project\\mental_health_bot\\.venv\\Scripts\\python.exe'

In [4]:
from langgraph.graph import StateGraph,MessagesState,START,END
from langgraph.graph.message import add_messages
from typing import Annotated,Literal,TypedDict
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from openai import RateLimitError


In [5]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage


class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    summary: str
    trimmed: list[AnyMessage]
    turns_since_summary: int

In [6]:
@tool
def search(query:str):
    "This is my tool"
    if "sf" in query.lower() or "san francisco" in query.lower():
        return "Its 60 degrees and frozen"
    return "Its 90 degree and sunny"

In [7]:
tools = [search]

In [8]:
tool_node = ToolNode(tools)
tool_node

tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'search': StructuredTool(name='search', description='This is my tool', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x000001EFADCDCB80>)}, _injected_args={'search': _InjectedArgs(state={}, store=None, runtime=None)}, _handle_tool_errors=<function _default_handle_tool_errors at 0x000001EFB039D080>, _messages_key='messages', _wrap_tool_call=None, _awrap_tool_call=None)

In [9]:
llm_with_tool = llm.bind_tools(tools)

In [10]:
def router_function(state:MessagesState)-> Literal["tools",END]:
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

In [11]:
def trim_messages(state:ChatState):
    MAX_MESSAGES = 12
    messages = state["messages"]
    # Actually use the trimmed list going forward
    trimmed_messages = messages[-MAX_MESSAGES:] if len(messages) > MAX_MESSAGES else messages

    return {
        "messages": messages,
        "trimmed": trimmed_messages,
    }

In [12]:
def summarize_if_needed(state:ChatState):
    SUMMARY_UPDATE_EVERY = 3
    messages = state["messages"]
    trimmed_messages = state.get("trimmed", [])
    summary = state.get("summary", "")
    turns = state.get("turns_since_summary", 0)
    
      # Increment turn counter
    turns += 1

    # Not time yet → skip summarization
    if turns < SUMMARY_UPDATE_EVERY:
        return {
            "messages": messages,
            "trimmed": trimmed_messages,
            "summary": summary,
            "turns_since_summary": turns
        }

    if len(messages) < 3:
        return {
        "messages": messages,
        "trimmed": trimmed_messages,
        "summary": summary,
        "turns_since_summary": turns
    }

    summary_prompt = [
        ("system", 
         "You are updating a conversation summary. Below is the existing summary. "
         "Review the following new messages and create an UPDATED cumulative summary that includes "
         "all important information from both the old summary and new messages. "
         "Keep it concise but comprehensive.\n\n"
         f"Existing summary: {summary}"),
        *trimmed_messages
    ]

    
# summary_prompt.extend(trimmed_messages)
    try:
        summary_response = llm.invoke(summary_prompt)
        new_summary = summary_response.content
    except RateLimitError:
        return state 

    return {
        "messages": messages,          # keep trimmed messages
        "trimmed":trimmed_messages,
        "summary": summary_response.content,
        "turns_since_summary": 0   # reset counter
    }



In [13]:
def call_model(state):
    messages = state["messages"]  
    trimmed_messages = state["trimmed"]    
    summary = state.get("summary", "")
    turns = state.get("turns_since_summary", 0)
    final_prompt = []

    if summary:
        final_prompt.append(
            ("system", f"Conversation summary so far: {summary}")
        )

    final_prompt.extend(trimmed_messages)

    response = llm_with_tool.invoke(final_prompt)

    return {
        "messages": [response],
        "turns_since_summary": turns   # increment counter
        
    }

In [14]:
workflow = StateGraph(ChatState)

workflow.add_node("trim", trim_messages)
workflow.add_node("summarize", summarize_if_needed)
workflow.add_node("agent", call_model)
workflow.add_node("tools",tool_node)

workflow.add_edge(START, "trim")
workflow.add_edge("trim", "summarize")
workflow.add_edge("summarize", "agent")
workflow.add_conditional_edges("agent",router_function,{"tools":"tools",END:END})
workflow.add_edge("tools","agent")


In [15]:
import sqlite3

class ChatStorage:
    def __init__(self):
        self.conn = sqlite3.connect("demomemory.db", check_same_thread=False)

    def save_message(self, thread_id, role, message):
        self.conn.execute(
            "INSERT INTO chat_history (thread_id, role, message) VALUES (?, ?, ?)",
            (thread_id, role, message)
        )
        self.conn.commit()

In [16]:
# app = workflow.compile()
# result = app.invoke(
#         {"messages": [("user", "what should i call you")]},
#     )
# result

In [19]:
from langgraph.checkpoint.sqlite import SqliteSaver

config = {"configurable": {"thread_id": "1"}}
storage = ChatStorage()   # initialize storage ONCE
DB_PATH = r"E:\CLg Acadamics\Mega Project\mental_health_bot\Backend_Langraph\DemoMemory.db"

with SqliteSaver.from_conn_string(DB_PATH) as memory:
    app = workflow.compile(checkpointer=memory)
    

    # runtime loop starts here
    while True:
        user_input = input("User: ")
        if user_input.lower() == "exit":
            
            break

        # save user message
        storage.save_message("1", "user", user_input)  
        result = app.invoke(
            {"messages": ("user", user_input)},
            config
        )

        # save bot reply
        bot_reply = result["messages"][-1].content
        storage.save_message("1", "assistant", bot_reply)
        print("Bot:", bot_reply)

        summary = result["summary"]
        print("summary:",summary)
        




Bot: Hi! It's nice to chat with you.

I've updated our conversation summary:

* Your favorite fruit is mango!
* Your name is not Jay!
* You haven't mentioned your favorite color (yet!)
* You also mentioned "Jay" a few times, but we've already established that's not your name.

Feel free to share anything you'd like, or ask a question!

(By the way, I've noticed you're saying "hi" again. Is there something specific you'd like to talk about or ask?)
summary: Hi!

I've updated our conversation summary:

* Your favorite fruit is mango!
* Your name is not Jay!
* You haven't mentioned your favorite color (yet!)
* You also mentioned "Jay" a few times, but we've already established that's not your name.

Feel free to share anything you'd like, or ask a question!
Bot: According to our conversation summary, your favorite fruit is... MANGO!
summary: I'm happy to remind you!

According to our conversation summary, your favorite fruit is... MANGO!


In [18]:
# def custom_add_messages(left: list[AnyMessage], right: Union[list[AnyMessage], tuple]) -> list[AnyMessage]:
#     """Custom message reducer that handles tuples by converting them to lists"""
#     if isinstance(right, tuple):
#         right = [right]
#     return add_messages(left, right)

# class ChatState(TypedDict):
#     messages: Annotated[list[AnyMessage], custom_add_messages]  # Use custom reducer
#     summary: str
#     trimmed: list[AnyMessage]

sqlite3 demomemory.db;
